# Figure 1: scIDiff Framework Overview

**Manuscript**: scIDiff - Learning Single-Cell Regulatory Dynamics with Hybrid Drift Fields  
**Journal**: Nature Computational Science  
**Repository**: https://github.com/manarai/scIDiff_V2

---

## Overview

This notebook generates **Figure 1** for the scIDiff manuscript, illustrating the complete framework:

- **Panel a**: Drift field learning from single-cell data in latent space
- **Panel b**: Jacobian eigenmode analysis over pseudotime
- **Panel c**: Temporal Jacobian tensor visualization
- **Panel d**: Archetype decomposition with activation profiles

All formatting follows **Nature Computational Science** publication standards (600 DPI, proper typography, colorblind-friendly palette).

---

## Installation

Before running this notebook, install the required packages:

```bash
# Clone the repository
git clone https://github.com/manarai/scIDiff_V2.git
cd scIDiff_V2

# Install the package
pip install -e .
```

## 1. Setup and Configuration

In [ ]:
# Import required libraries
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from mpl_toolkits.mplot3d import Axes3D

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

print("✓ Libraries imported successfully")

In [ ]:
# Configure matplotlib for Nature Computational Science standards
plt.rcParams['figure.dpi'] = 600
plt.rcParams['savefig.dpi'] = 600
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 7
plt.rcParams['axes.linewidth'] = 0.5
plt.rcParams['xtick.major.width'] = 0.5
plt.rcParams['ytick.major.width'] = 0.5
plt.rcParams['xtick.major.size'] = 2
plt.rcParams['ytick.major.size'] = 2
plt.rcParams['xtick.labelsize'] = 6
plt.rcParams['ytick.labelsize'] = 6
plt.rcParams['axes.labelsize'] = 7
plt.rcParams['legend.fontsize'] = 6
plt.rcParams['pdf.fonttype'] = 42  # TrueType fonts for PDF
plt.rcParams['ps.fonttype'] = 42

print("✓ Nature Computational Science formatting applied")
print("  • Panel labels: 8pt bold")
print("  • Axis labels: 7pt")
print("  • Equations: 6pt")
print("  • Resolution: 600 DPI")
print("  • Width: 180mm")

## 2. Generate Synthetic Data

We create synthetic single-cell data with a branching trajectory to demonstrate the scIDiff framework.

In [ ]:
# Data parameters
n_cells = 500
n_time_points = 100
latent_dim = 10
n_archetypes = 4

print(f"Generating synthetic data:")
print(f"  • {n_cells} cells")
print(f"  • {n_time_points} time points")
print(f"  • {latent_dim} latent dimensions")
print(f"  • {n_archetypes} regulatory archetypes")

In [ ]:
def generate_branching_trajectory(n_points=500):
    """
    Generate a branching trajectory in 2D latent space.
    
    Returns:
        t: pseudotime values
        x_main, y_main: main trajectory before branching
        x_b1, y_b1: branch 1 coordinates
        x_b2, y_b2: branch 2 coordinates
    """
    t = np.linspace(0, 1, n_points)
    x_main = t * 2 - 1
    y_main = np.zeros_like(t)
    
    # Branch at t=0.5
    branch_idx = int(n_points * 0.5)
    
    # Branch 1 (upward)
    x_branch1 = x_main.copy()
    y_branch1 = y_main.copy()
    y_branch1[branch_idx:] = (t[branch_idx:] - 0.5) * 2
    
    # Branch 2 (downward)
    x_branch2 = x_main.copy()
    y_branch2 = y_main.copy()
    y_branch2[branch_idx:] = -(t[branch_idx:] - 0.5) * 2
    
    return t, x_main, y_main, x_branch1, y_branch1, x_branch2, y_branch2

# Generate trajectory
t, x_main, y_main, x_b1, y_b1, x_b2, y_b2 = generate_branching_trajectory(n_cells)
pseudotime = t

# Create cell positions with biological noise
cells_x = np.concatenate([x_main[:250], x_b1[250:375], x_b2[375:]])
cells_y = np.concatenate([y_main[:250], y_b1[250:375], y_b2[375:]])
cells_x += np.random.normal(0, 0.05, n_cells)
cells_y += np.random.normal(0, 0.05, n_cells)

print("✓ Branching trajectory generated")

In [ ]:
def compute_drift_field(X, Y):
    """
    Compute drift field vectors for visualization.
    
    Args:
        X, Y: meshgrid coordinates
    
    Returns:
        U, V: drift components
    """
    U = np.ones_like(X) * 0.5
    V = np.zeros_like(Y)
    
    # Add branching dynamics
    branch_region = (X > 0) & (X < 0.5)
    V[branch_region & (Y > 0)] = 0.3
    V[branch_region & (Y < 0)] = -0.3
    
    return U, V

# Create drift field grid
x_grid = np.linspace(-1.2, 1.2, 15)
y_grid = np.linspace(-1.2, 1.2, 15)
X_grid, Y_grid = np.meshgrid(x_grid, y_grid)
U_grid, V_grid = compute_drift_field(X_grid, Y_grid)

print("✓ Drift field computed")

In [ ]:
# Generate temporal Jacobian tensor
jacobian_tensor = np.zeros((n_time_points, latent_dim, latent_dim))

for t_idx in range(n_time_points):
    J = np.random.randn(latent_dim, latent_dim) * 0.1
    J = (J + J.T) / 2  # Make symmetric
    jacobian_tensor[t_idx] = J

# Compute eigenvalues over time
eigenvalues_over_time = np.zeros((n_time_points, latent_dim))

for t_idx in range(n_time_points):
    eigvals = np.linalg.eigvals(jacobian_tensor[t_idx])
    eigenvalues_over_time[t_idx] = np.sort(eigvals.real)[::-1]

print("✓ Jacobian tensor and eigenvalues computed")

In [ ]:
# Generate archetype activation profiles
activation_profiles = np.zeros((n_archetypes, n_time_points))
time_axis = np.linspace(0, 1, n_time_points)

# Archetype 1: Early activation (Gaussian)
activation_profiles[0] = np.exp(-((time_axis - 0.2) ** 2) / 0.05)

# Archetype 2: Mid activation (Gaussian)
activation_profiles[1] = np.exp(-((time_axis - 0.5) ** 2) / 0.03)

# Archetype 3: Late activation (Sigmoid)
activation_profiles[2] = 1 / (1 + np.exp(-20 * (time_axis - 0.6)))

# Archetype 4: Very late activation (Sigmoid)
activation_profiles[3] = 1 / (1 + np.exp(-20 * (time_axis - 0.7)))

print("✓ Archetype activation profiles generated")

## 3. Create Figure 1

Generate the four-panel figure with proper Nature Computational Science formatting.

In [ ]:
# Create figure (180mm width = 7.09 inches)
fig = plt.figure(figsize=(7.09, 5.5))
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.4,
                       left=0.08, right=0.98, top=0.96, bottom=0.08)

# Colorblind-friendly palette
color_unstable = '#D55E00'  # vermillion
color_stable = '#0072B2'    # blue
color_weakly = '#999999'    # gray
colors_archetypes = ['#E69F00', '#56B4E9', '#009E73', '#F0E442']  # orange, sky blue, green, yellow

### Panel a: Drift Field Learning

In [ ]:
ax_a = fig.add_subplot(gs[0, 0])

# Plot cells colored by pseudotime
ax_a.scatter(cells_x[:250], cells_y[:250], c=pseudotime[:250], 
            cmap='viridis', s=8, alpha=0.6, edgecolors='none', rasterized=True)
ax_a.scatter(cells_x[250:375], cells_y[250:375], c=pseudotime[250:375], 
            cmap='viridis', s=8, alpha=0.6, edgecolors='none', rasterized=True)
ax_a.scatter(cells_x[375:], cells_y[375:], c=pseudotime[375:], 
            cmap='viridis', s=8, alpha=0.6, edgecolors='none', rasterized=True)

# Plot drift field (vector field)
ax_a.quiver(X_grid, Y_grid, U_grid, V_grid, 
           alpha=0.35, scale=8, width=0.002, color='black', headwidth=4, headlength=5)

# Plot trajectory lines
ax_a.plot(x_main[:250], y_main[:250], 'k-', linewidth=1.5, alpha=0.5)
ax_a.plot(x_b1[250:], y_b1[250:], 'k-', linewidth=1.5, alpha=0.5)
ax_a.plot(x_b2[250:], y_b2[250:], 'k-', linewidth=1.5, alpha=0.5)

# Formatting
ax_a.set_xlabel('Latent dimension 1', fontsize=7)
ax_a.set_ylabel('Latent dimension 2', fontsize=7)
ax_a.set_title('a', fontsize=8, fontweight='bold', loc='left', pad=3)
ax_a.set_xlim(-1.3, 1.3)
ax_a.set_ylim(-1.3, 1.3)
ax_a.set_aspect('equal')
ax_a.tick_params(labelsize=6)

# Add SDE equation
ax_a.text(0.05, 0.97, r'$\mathrm{d}X_t = f(X_t, t)\mathrm{d}t + \sqrt{2\beta}\,\mathrm{d}W_t$', 
         transform=ax_a.transAxes, fontsize=6, verticalalignment='top',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.9, 
                  edgecolor='gray', linewidth=0.5))

print("✓ Panel a: Drift field learning")

### Panel b: Jacobian Eigenmodes

In [ ]:
ax_b = fig.add_subplot(gs[0, 1])

time_plot = np.linspace(0, 1, n_time_points)

# Plot eigenvalue trajectories
ax_b.plot(time_plot, eigenvalues_over_time[:, 0], color=color_unstable, 
         linewidth=1.2, label='Unstable', alpha=0.9)
ax_b.plot(time_plot, eigenvalues_over_time[:, 1], color=color_weakly, 
         linewidth=1.2, label='Weakly stable', alpha=0.7, linestyle='--')
ax_b.plot(time_plot, eigenvalues_over_time[:, 2], color=color_stable, 
         linewidth=1.2, label='Stable', alpha=0.9)

# Add reference lines
ax_b.axhline(y=0, color='black', linestyle=':', linewidth=0.5, alpha=0.5)
ax_b.axvline(x=0.5, color='#CC0000', linestyle='--', linewidth=0.8, alpha=0.4)
ax_b.text(0.51, ax_b.get_ylim()[1] * 0.88, 'Branch\npoint', 
         ha='left', fontsize=5.5, color='#CC0000')

# Formatting
ax_b.set_xlabel('Pseudotime', fontsize=7)
ax_b.set_ylabel('Eigenvalue (λ)', fontsize=7)
ax_b.set_title('b', fontsize=8, fontweight='bold', loc='left', pad=3)
ax_b.legend(fontsize=6, frameon=True, loc='upper right', framealpha=0.9)
ax_b.grid(True, alpha=0.15, linewidth=0.3)
ax_b.tick_params(labelsize=6)

# Add Jacobian equation
ax_b.text(0.05, 0.08, r'$J(t) = \nabla_{\!x} f(x(t), t)$', 
         transform=ax_b.transAxes, fontsize=6, verticalalignment='bottom',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.9, 
                  edgecolor='gray', linewidth=0.5))

print("✓ Panel b: Jacobian eigenmodes")

### Panel c: Temporal Jacobian Tensor

In [ ]:
ax_c = fig.add_subplot(gs[1, 0], projection='3d')

# Select time slices to visualize
n_slices = 10
slice_indices = np.linspace(0, n_time_points-1, n_slices, dtype=int)

# Plot stacked Jacobian matrices
for i, t_idx in enumerate(slice_indices):
    J_slice = jacobian_tensor[t_idx, :5, :5]  # Show 5x5 subset
    
    x = np.arange(5)
    y = np.arange(5)
    X, Y = np.meshgrid(x, y)
    Z = np.ones_like(X) * i * 2
    
    # Color by Jacobian values
    colors = plt.cm.RdBu_r((J_slice - J_slice.min()) / (J_slice.max() - J_slice.min() + 1e-10))
    ax_c.plot_surface(X, Y, Z, facecolors=colors, alpha=0.65, 
                     shade=False, linewidth=0.1, edgecolor='gray', antialiased=True)

# Formatting
ax_c.set_xlabel('Gene $i$', fontsize=6, labelpad=1)
ax_c.set_ylabel('Gene $j$', fontsize=6, labelpad=1)
ax_c.set_zlabel('Time', fontsize=6, labelpad=1)
ax_c.set_title('c', fontsize=8, fontweight='bold', loc='left', pad=3)
ax_c.view_init(elev=20, azim=45)
ax_c.tick_params(labelsize=5, pad=-2)

# Add tensor notation
ax_c.text2D(0.05, 0.97, r'$\mathcal{J} \in \mathbb{R}^{T \times d \times d}$', 
           transform=ax_c.transAxes, fontsize=6, verticalalignment='top',
           bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.9, 
                    edgecolor='gray', linewidth=0.5))

print("✓ Panel c: Temporal Jacobian tensor")

### Panel d: Archetype Decomposition

In [ ]:
ax_d = fig.add_subplot(gs[1, 1])

# Plot archetype activation profiles
for k in range(n_archetypes):
    ax_d.plot(time_axis, activation_profiles[k], color=colors_archetypes[k], 
             linewidth=1.5, label=f'Archetype {k+1}', alpha=0.85)
    
    # Mark peak activation
    peak_idx = np.argmax(activation_profiles[k])
    ax_d.plot(time_axis[peak_idx], activation_profiles[k, peak_idx], 
             'o', color=colors_archetypes[k], markersize=4)

# Formatting
ax_d.set_xlabel('Pseudotime', fontsize=7)
ax_d.set_ylabel('Activation $a_k(t)$', fontsize=7)
ax_d.set_title('d', fontsize=8, fontweight='bold', loc='left', pad=3)
ax_d.legend(fontsize=6, frameon=True, loc='upper left', ncol=2, framealpha=0.9,
           columnspacing=1, handlelength=1.5)
ax_d.grid(True, alpha=0.15, linewidth=0.3)
ax_d.set_ylim(-0.1, 1.2)
ax_d.tick_params(labelsize=6)

# Add decomposition equation
ax_d.text(0.98, 0.08, r'$J(t) \approx \sum_{k=1}^{K} a_k(t) M_k$', 
         transform=ax_d.transAxes, fontsize=6, verticalalignment='bottom',
         horizontalalignment='right',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.9, 
                  edgecolor='gray', linewidth=0.5))

# Annotate sequential handoff
ax_d.annotate('', xy=(0.55, 0.75), xytext=(0.45, 0.75),
             arrowprops=dict(arrowstyle='<->', color='#CC0000', lw=1.2, alpha=0.6))
ax_d.text(0.5, 0.80, 'Sequential\nhandoff', ha='center', fontsize=5.5, color='#CC0000')

print("✓ Panel d: Archetype decomposition")

## 4. Save Figure

In [ ]:
# Save in both PNG and PDF formats
plt.savefig('Figure1_scIDiff.png', dpi=600, bbox_inches='tight')
plt.savefig('Figure1_scIDiff.pdf', dpi=600, bbox_inches='tight')

print("\n" + "="*70)
print("Figure 1 generated successfully!")
print("="*70)
print("\nOutput files:")
print("  • Figure1_scIDiff.png (600 DPI, raster)")
print("  • Figure1_scIDiff.pdf (600 DPI, vector)")
print("\nBoth formats are ready for Nature Computational Science submission.")
print("="*70)

plt.show()

## Summary

This notebook successfully generates **Figure 1** for the scIDiff manuscript with the following features:

### Nature Computational Science Compliance

**Typography**:
- Panel labels (a, b, c, d): 8pt bold sans-serif ✓
- Axis labels: 7pt sans-serif (Helvetica/Arial) ✓
- Equations: 6pt ✓
- Legend and tick labels: 6pt ✓

**Technical Specifications**:
- Resolution: 600 DPI (exceeds 300 DPI minimum) ✓
- Width: 180mm (7.09 inches) ✓
- Font family: Helvetica/Arial sans-serif ✓
- Font embedding: TrueType (Type 42) for PDF ✓
- Color palette: Colorblind-friendly ✓

### Figure Components

- **Panel a**: Drift field learning with SDE equation dXₜ = f(Xₜ, t)dt + √(2β)dWₜ
- **Panel b**: Jacobian eigenmodes showing unstable, weakly stable, and stable modes
- **Panel c**: Temporal Jacobian tensor 𝒥 ∈ ℝ^(T×d×d) as stacked 3D matrices
- **Panel d**: Archetype decomposition J(t) ≈ Σ aₖ(t)Mₖ with sequential handoff

---

**Repository**: https://github.com/manarai/scIDiff_V2  
**Package**: `scqdiff` v0.1.0  
**Citation**: [Add manuscript citation when published]

For questions or issues, please open an issue on GitHub.